# Final Project: Kalshi Market Making in short-dated SPX Events

Chris Mulligan (12502987), George Lord (12243747), Max Zhalilo (12341701)

# Content

0. Abstract
1. Introduction
2. Data
3. Techniques and Theory Details in the Strategy
4. Implementation of our Strategy
5. Result/Analysis

# 0. Abstract

We use statistical modeling & black-scholes to price Kalshi short-dates SPX events. We simulate an automated market making strategy in these events using a simulator that directly interfaces with our trading modules. We delta hedge SPX exposure by trading the SPY ETF. We test the performance of our strategy over 2 months.

# 1. Introduction

In modern financial markets, algorithmic trading and automated market making have become central components of liquidity provision and price discovery. Electronic Market Makers such as Jump & Virtu have begun to expand into prediction markets such as Kalshi & Polymarket to profit from their increasing trading volumes. In this project, we develop and test a fully systematic market-making strategy for short-dated S&P 500 (SPX) event contracts listed on Kalshi. These contracts pay a fixed $1 if a specified event occurs—such as the SPX closing above a certain level-and $0 otherwise. As a result, they can be interpreted as binary options on the terminal value of the SPX index.

The core idea of our strategy is grounded in Black-Scholes derivative pricing theory. A binary call option has a value equal to the discounted risk-neutral probability that the underlying asset finishes above a given strike at expiration. For contracts with 0–1 days to expiration, discounting effects are negligible (??? LETS THINK ABOUT THIS ???), so the contract price closely approximates the market-implied probability of the event occurring. By modeling the terminal distribution of SPX under a log-normal assumption consistent with the Black–Scholes framework, we can estimate these probabilities analytically using closed-form expressions. Specifically, we compute the cumulative distribution function (CDF) implied by current SPX levels, time to expiration, and volatility, with volatility proxied by the VIX index.

Once fair values for binary calls and puts are obtained, we replicate the payoff structures of Kalshi’s SPX “above” and “range” contracts using combinations of these binaries. This replication framework provides a theoretical benchmark price for each listed contract. The strategy then centers its market-making quotes around these fair values and dynamically adjusts bid–ask spreads as a function of market volatility, time to expiration, tick size constraints, exchange fees, and current inventory. In higher-volatility or low-liquidity environments, spreads are widened to compensate for increased uncertainty and adverse selection risk.

Because the fair value of each contract is sensitive to movements in the underlying SPX index, holding inventory in Kalshi contracts introduces directional exposure. To manage this risk, we compute the delta of each replicated binary position with respect to SPX and aggregate the book’s total delta exposure. We then hedge this exposure using the SPY ETF, which closely tracks SPX movements. This delta-hedging process isolates the strategy’s profit and loss primarily to bid–ask spread capture and pricing inefficiencies rather than outright market direction.

We test our strategy in a simulator that mimics Kalshi exchange capability. We've built out our strategy using pricing, market making, hedging, & position management modules, all of which interact with the simulator using our execution engine module. Our objective is to assess whether theoretically grounded probability estimates, combined with disciplined spread management and delta hedging, can generate consistent risk-adjusted returns in short-dated event markets.

# 2. Data

Chris to-do?

# 3. Theory

George to-do?

# 4. Implementation

Max to-do?

## 4.1 Logic

### Pricing Logic:

### Market Making Logic: 

Base spread: FV +- .01 = .02. FV: Fair Value

The spread changes based off VIX & Current Inventory (Position Size):

VIX - For every point of VIX, the spread increases by .001. Ie. VIX = 20 -> spread increases by .02

Inventory - For every (Kalshi) contract in our inventory, the spread increases by .00025. Ie. 80 Contracts Held -> spread increases by .02

Outside of regular market hours (9:30 AM - 4:00 PM EST), the spread increases by .10 to account for the significantly lower liquidity & trade volume.

The spread midpoint is skewed away from the fair by (-).0005 * Kalshi Contract Position. Ie. Holding 40 "Yes" Contracts -> fair decreases by .02. This is done to reduce our axed position faster.

Finally, we build in Kalshi Maker Fees into our spread.

- Kalshi Maker Fees: fee = round up(0.0175 * C * P * (1 - P)), 
where P = price, C = number of contracts.

Our bids & offers are placed on the end of our spread.

We also manage our bid / ask size based on inventory (total & sided) to stay within the following parameters:
- Default Quote Size = 50
- Max Quote Size = 100
- Max Abs Inventory for Size = 500

### Simulation Logic:

The simulation seeks to mimic Kalshi taker trades. We assume last in queue position at our bid/ask prices. All of our data gets passed every second, so our simulation iterates through each second of our data. On each tick:
- We update our quote for each contract.
- Check for fills (bid_taken < our bid or ask_taken > our ask)
- Contracts resolve to be worth either 0 or $1 depending on outcome at expry.

### Delta Hedging Logic

## 4.2 Program Architecture

In our implementation, we created a modular market making bot that currently interfaces with our simulator model. That follows the design below:

![System Architecture](Data/architecture.jpg)

### NOTE: NEED TO REFRESH DRAWING

## Data Ingestor

- **What it does:** Loads and standardizes 1-second Kalshi and market data from parquet files into structured Polars DataFrames.
- **Takes in:** Paths to Kalshi parquet file(s) (per-contract or cleaned trade-event), plus SPX, VIX, and SPY parquet files, as specified by the ingest config.
- **Sends out:** `all_df` (long-form combined Kalshi + macro data) to the Simulator for the main run; `macro_df` (SPX/VIX/SPY only) used by the ExecutionEngine.


## Simulator

- **What it does:** Drives the simulation by iterating over each second in chronological order. Each tick it calls the ExecutionEngine to get resting quotes and  hedging trades; applies a deterministic last-in-queue fill rule (e.g. bid filled when take_bid ≤ our_bid − .01); sends resulting fill intents to the engine. Produces a per-second, per-contract log of market snapshots, bot quotes, fill flags, and position state.
- **Takes in:** `all_df` (from DataIngestor). Each tick it receives resting quotes and hedge orders from the ExecutionEngine via the engine’s `on_tick` return value and applies them when evaluating fills.
- **Sends out:** `FillIntent` objects to the ExecutionEngine when a fill is detected; and a Polars DataFrame (one row per ts × contract_id) with the full log of market and bot behavior for analysis.


## Execution Engine

- **What it does:** Central gateway that coordinates pricing, quoting, hedging, and position updates. Each tick it applies any delayed trades due at that time (including SPY hedges with a 1-second delay), settles expired Kalshi contracts, requests fair values from the Pricer, pushes fair values and positions to the MarketMaker to get quotes, and asks the Delta Hedger for a hedge order (which it then schedules with delay). Kalshi fills from the Simulator are applied to the PositionManager immediately.
- **Takes in:** SPX, VIX, and SPY (and contract list) each tick from the Simulator, which reads them from the ingested data; fill intents from the Simulator when fills occur; internally it receives quotes from the MarketMaker and hedge orders from the Delta Hedger (both invoked inside the engine).
- **Sends out:** Fair-value requests to the Pricer; VIX, fair values, and position state to the MarketMaker; Kalshi fills and (when due) SPY trades to the PositionManager; current resting quotes to the Simulator; and hedge orders to the Simulator (execution applied with 1-second delay via the PositionManager).

## Pricer

- **What it does:** Computes the fair value (a probability in [0, 1]) of the "YES" side of each Kalshi SPX contract using a Black–Scholes-style lognormal terminal distribution. For threshold contracts (KXINXU), it prices a single binary call; for range contracts (KXINX), it replicates the payoff as a call spread (long binary call at lower strike, short at upper strike). Volatility is taken from VIX/100; time to expiry is parsed from the contract ID.
- **Takes in:** `contract_id`, `spx`, `vix`, and `ts` (current timestamp in UTC), passed by the ExecutionEngine each tick.
- **Sends out:** A fair value per contract. The ExecutionEngine forwards these values to the MarketMaker and uses them for quoting.

## Market Maker

- **What it does:** Produces bid/ask quotes and sizes for each contract from the latest fair value, VIX, and position state. Spread is widened by base spread plus terms for VIX and absolute inventory; the mid is skewed away from fair value based on inventory to mean-revert (e.g. long inventory → lower mid to encourage selling). Sizes can be tilted (e.g. larger ask when long) and are reduced as inventory grows. Optionally widens spread outside regular market hours (9:30–16:00 ET). Prices are rounded to tick size and clamped to [0, 1].
- **Takes in:** Fair value per contract (from Pricer via ExecutionEngine), VIX and market-hours flag (from ExecutionEngine), and per-contract inventory/cash (from PositionManager via ExecutionEngine).
- **Sends out:** A `Quote` per contract (contract_id, fair_value, bid, ask, bid_size, ask_size, spread, meta). The ExecutionEngine collects these and sends the resulting resting quotes to the Simulator.

## Position Manager

- **What it does:** Tracks Kalshi positions by contract, SPY position, and cash. Applies Kalshi fills (from the Simulator via ExecutionEngine) immediately, updating inventory and cash. Applies SPY hedge trades when their delayed execution time is reached. At each tick, settles any Kalshi contracts that have expired (settlement at start of next day after expiry) by paying out 0 or 1 per contract and removing them from the book.
- **Takes in:** Kalshi trade events (contract_id, side, qty, price) from ExecutionEngine when fills occur; SPY trade events (side, qty, price) when delayed hedge trades execute; current timestamp and settlement SPX for deciding which contracts to settle.
- **Sends out:** Per-contract Kalshi inventory, total SPY position, and cash balance. The ExecutionEngine reads these (e.g. `get_kalshi_position`, `get_kalshi_positions`, `get_spy_position`, `get_cash`) and passes them to the MarketMaker and Delta Hedger; it also uses them when applying fills and scheduling hedge trades.

## Delta Hedger

- **What it does:** Computes the aggregate delta of the Kalshi book in SPX terms using the binary-option delta formula (φ(d2) / (S × σ × √τ)) per contract, then maps that to a target SPY share position via a configurable SPX-to-SPY scaling (e.g. dSPY/dSPX ≈ 0.1 → shares ≈ −10 × delta_book_spx). Produces a single hedge order (buy or sell SPY) to move the current SPY position toward the target, with optional cap on size per tick.
- **Takes in:** Current timestamp `ts`, `spy_price`, `spx_price`, `vix`, full `kalshi_positions` (contract_id → qty), and `current_spy_position`, all provided by the ExecutionEngine each tick from market data and the PositionManager.
- **Sends out:** A `HedgeOrder` (ts, symbol, side, qty, ref_price) or `None` if no rehedge is needed. The ExecutionEngine schedules the trade with a 1-second execution delay and later sends it to the Simulator; when the delay elapses, the engine applies the trade through the PositionManager.

# 5. Results & Analysis